AutoModelForCausalLM - actual "brain" that predicts the next word to generate an answer.

In [ ]:
# Deep Learning & AI Model Loading
import torch #handles the heavy mathematical calculation
import transformers   #ownloading, loading, and running pre-trained AI models.
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM 

#Reading and Preparing Documents
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter

#Vector Search (The "Retrieval" in RAG)
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS  #specifically for storing and searching vectors.
from langchain.chains import ConversationalRetrievalChain
from langchain_community.llms import HuggingFacePipeline

#User Interface & System Tools
import gradio as gr
import os

print("Libraries imported successfully!")


W0608 11:06:36.940000 25432 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Libraries imported successfully!


In [7]:
import os

# 1. Define the EXACT name of your file
PDF_FILENAME = "pythonlearn (2).pdf"

# 2. Define ONLY the folder path (DO NOT put the filename at the end here!)
DOWNLOADS_FOLDER = r"C:\Users\KHUSHI\Downloads"

# 3. Combine them correctly using os.path.join
DOCUMENT_PATH = os.path.join(DOWNLOADS_FOLDER, PDF_FILENAME)

# 4. Let's test if Python can actually see it
print(f"🔍 Looking for file at: {DOCUMENT_PATH}")

if os.path.exists(DOCUMENT_PATH):
    print(f"✅ SUCCESS! File found.")
else:
    print(f"❌ ERROR: File not found. Please check your Downloads folder to ensure the name is exactly '{PDF_FILENAME}'")

🔍 Looking for file at: C:\Users\KHUSHI\Downloads\pythonlearn (2).pdf
✅ SUCCESS! File found.


In [10]:
# 1. Import the correct loader for PDFs
from langchain_community.document_loaders import PyPDFLoader

# 2. Load the PDF document (DO NOT use TextLoader for PDFs)
loader = PyPDFLoader(DOCUMENT_PATH)
documents = loader.load()

print(f"✅ Successfully loaded {len(documents)} pages from the PDF.")

# 3. Split the document into chunks
# Note: RecursiveCharacterTextSplitter is generally better than CharacterTextSplitter 
# because it tries to split at paragraph/sentence boundaries first.
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""] # Splits smartly by paragraphs, then sentences
)
texts = text_splitter.split_documents(documents)

print(f"✅ Split document into {len(texts)} searchable chunks.")

# 4. Create embeddings (CPU is fine, as you specified)
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", 
    model_kwargs={'device': 'cpu'}
)

# 5. Create and save the FAISS vector store
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(texts, embeddings)
db.save_local(FAISS_DB_PATH)
print(f"✅ FAISS index created and saved to: {FAISS_DB_PATH}")

✅ Successfully loaded 241 pages from the PDF.
✅ Split document into 583 searchable chunks.


c:\Users\KHUSHI\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ FAISS index created and saved to: data\faiss_index


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
import torch

# Load a smaller conversational model suitable for CPU
model_name = "microsoft/DialoGPT-small" # Lighter model for CPU

print("⏳ Loading tokenizer and model...")
# Load tokenizer and model (removed device_map to avoid accelerate conflict)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32 # Use float32 for CPU stability
)

print("⏳ Setting up Hugging Face pipeline...")
# Create a Hugging Face pipeline for text generation
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50,       # Limit response length for CPU performance
    temperature=0.7,
    top_p=0.95,
    do_sample=True,
    device='cpu',            # Explicitly tell the pipeline to use CPU
)

# Create a HuggingFacePipeline LLM for LangChain
llm = HuggingFacePipeline(pipeline=pipe)
print("✅ Language model loaded and pipeline set up successfully!")

⏳ Loading tokenizer and model...
⏳ Setting up Hugging Face pipeline...
✅ Language model loaded and pipeline set up successfully!


In [13]:
# Setup the conversational retrieval chain
# We will use 'stuff' chain type for simplicity, which combines all documents into the prompt.
# For larger contexts, 'map_reduce', 'refine', or 'map_rerank' might be more suitable.
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=db.as_retriever(),
    return_source_documents=True
)
print("Conversational Retrieval Chain set up.")


Conversational Retrieval Chain set up.


In [14]:
chat_history = []
question = "What did the president say about the economy?"
result = qa_chain({"question": question, "chat_history": chat_history})
print(f"Question: {question}")
print(f"Answer: {result['answer']}")

# Update chat history for the next turn
chat_history = [(question, result["answer"])]

question = "What about healthcare?"
result = qa_chain({"question": question, "chat_history": chat_history})
print(f"Question: {question}")
print(f"Answer: {result['answer']}")


c:\Users\KHUSHI\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Question: What did the president say about the economy?
Answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

on a farm and walking around behind a barn, with no sign of a restaurant.
Then you say “did you turn left or right at the gas station?” and they say, “ I
followed your directions perfectly, I have them written down, it says turn left
and go one mile at the gas station. ” Then you say, “I am very sorry, because

in the book you will look back and see that it was all really easy and elegan t and
it simply took you a bit of time to absorb it.
1.13 Glossary
bug An error in a program.
central processing unit The heart of any computer. It is what runs the software
that we write; also called “CPU” or “the processor” .

18 CHAPTER 1. WHY SHOULD YOU LEARN TO WRITE PROGRAMS?

216 CHAPTER 16. VISUALIZING DATA

Question: What did the president say about the economy?
Helpful

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Question: What about healthcare?
Answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

on a farm and walking around behind a barn, with no sign of a restaurant.
Then you say “did you turn left or right at the gas station?” and they say, “ I
followed your directions perfectly, I have them written down, it says turn left
and go one mile at the gas station. ” Then you say, “I am very sorry, because

in the book you will look back and see that it was all really easy and elegan t and
it simply took you a bit of time to absorb it.
1.13 Glossary
bug An error in a program.
central processing unit The heart of any computer. It is what runs the software
that we write; also called “CPU” or “the processor” .

2 CHAPTER 1. WHY SHOULD YOU LEARN TO WRITE PROGRAMS?
Pick 
Me! 
Pick 
Me! 
Pick 
Me! 
Pick 
Me! 
Pick 
Me! 
Buy 
Me :) 
Figure 1.2: Programmers Talking to You
For example, loo

In [17]:
def predict(message, history):
    # Convert Gradio chat history to LangChain chat history format
    langchain_history = []
    for human, ai in history:
        langchain_history.append((human, ai))

    # Get response from the QA chain
    # The chain needs 'question' and 'chat_history'
    result = qa_chain({"question": message, "chat_history": langchain_history})
    response = result["answer"]
    return response

# Create the Gradio interface
iface = gr.ChatInterface(
    predict,
    chatbot=gr.Chatbot(height=300),
    textbox=gr.Textbox(placeholder="Ask me a question about the document...", container=False, scale=7),
    title="RAG Chatbot using LangChain and HuggingFace, CPU Only",
    description="Ask questions about your uploaded document.",
    # theme="soft",  <--- REMOVED THIS LINE TO FIX THE ERROR
    examples=["What is the main topic?", "What challenges were mentioned?", "What did he say about unity?"]
)

# Launch the interface
iface.launch(share=False, inbrowser=True) # Added inbrowser=True to auto-open in VS Code

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
